In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformer_lens
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import tqdm
from functools import partial

In [4]:
from sae import SparseAutoEncoder

In [2]:
model = transformer_lens.HookedTransformer.from_pretrained("pythia-70m")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-70m into HookedTransformer


In [24]:
SAE = SparseAutoEncoder(d=512, m=4096)
SAE.load_state_dict(torch.load("saved_models/sae_model_8x.pt", map_location="cpu"))
SAE.eval()

device = model.cfg.device
SAE.to(device)

SparseAutoEncoder(
  (act): ReLU()
)

In [25]:
def ablation_hook(resid, hook, feature_idx):
    f = SAE.encoder(resid)
    contr = f[..., feature_idx].unsqueeze(-1) * SAE.W_dec.data[feature_idx]
    return resid - contr

In [26]:
text = "LONDON (Reuters) - The British government announced"

In [27]:
IDX = 3440

ablated_logits = model.run_with_hooks(
    text,
    fwd_hooks=[("blocks.3.hook_resid_post", partial(ablation_hook, feature_idx=IDX))]
)

In [28]:
clean_logits = model(text)

In [37]:
clean_logprobs = F.log_softmax(clean_logits.float(), dim=-1)
ablated_logprobs = F.log_softmax(ablated_logits.float(), dim=-1)
clean_probs = clean_logprobs.exp()

kl_div = (clean_probs * (clean_logprobs - ablated_logprobs)).sum(dim=-1)
print(f"KL divergence per position: {kl_div.squeeze()}")
print(f"Mean KL divergence: {kl_div.mean().item():.6f}")

clean_top = clean_logits.argmax(dim=-1).squeeze()
ablated_top = ablated_logits.argmax(dim=-1).squeeze()

tokenizer = model.tokenizer
for pos in range(len(clean_top)):
    if clean_top[pos] != ablated_top[pos]:
        print(f"Position {pos}: {tokenizer.decode(clean_top[pos])} → {tokenizer.decode(ablated_top[pos])}")

KL divergence per position: tensor([0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0944e-03, 2.5538e-01,
        2.1655e-03, 2.7188e-04, 6.8605e-05, 5.0217e-06, 2.2139e-05, 2.0424e-05],
       device='mps:0', grad_fn=<SqueezeBackward0>)
Mean KL divergence: 0.021586


In [38]:
tokens = model.to_str_tokens(text)

# KL per token position
print("KL divergence per token:\n")
for pos, (tok, kl) in enumerate(zip(tokens, kl_div.squeeze())):
    print(f"{pos:2d} {tok:>15s}  KL={kl:.6f}")

# Top-5 predictions comparison
k = 5
print("\nTop-5 predictions (clean vs ablated):\n")
for pos in range(len(tokens)):
    clean_topk = clean_logits[0, pos].topk(k)
    ablated_topk = ablated_logits[0, pos].topk(k)
    print(f"Position {pos}: '{tokens[pos]}'")
    print(f"  Clean:   {[tokenizer.decode(t) for t in clean_topk.indices]}")
    print(f"  Ablated: {[tokenizer.decode(t) for t in ablated_topk.indices]}")
    print()

KL divergence per token:

 0   <|endoftext|>  KL=0.000000
 1               L  KL=0.000000
 2             OND  KL=0.000000
 3              ON  KL=0.000000
 4               (  KL=0.001094
 5         Reuters  KL=0.255378
 6               )  KL=0.002166
 7               -  KL=0.000272
 8             The  KL=0.000069
 9         British  KL=0.000005
10      government  KL=0.000022
11       announced  KL=0.000020

Top-5 predictions (clean vs ablated):

Position 0: '<|endoftext|>'
  Clean:   ['Q', 'The', '\n', 'A', '1']
  Ablated: ['Q', 'The', '\n', 'A', '1']

Position 1: 'L'
  Clean:   ['OND', 'incoln', 'ynn', 'ith', 'unch']
  Ablated: ['OND', 'incoln', 'ynn', 'ith', 'unch']

Position 2: 'OND'
  Clean:   ['ON', 'OND', 'ORE', 'ORD', 'AN']
  Ablated: ['ON', 'OND', 'ORE', 'ORD', 'AN']

Position 3: 'ON'
  Clean:   [',', ' (', ':', ' –', ' —']
  Ablated: [',', ' (', ':', ' –', ' —']

Position 4: ' ('
  Clean:   ['AP', 'Reuters', 'CBS', 'CNN', 'AFP']
  Ablated: ['AP', 'Reuters', 'CBS', 'CNN', 'AFP'

In [39]:
# Greedy generation: clean vs ablated
def generate_with_ablation(text, feature_idx, max_new_tokens=20):
    clean_tokens = [text]
    ablated_tokens = [text]

    clean_input = text
    ablated_input = text

    for _ in range(max_new_tokens):
        clean_logits = model(clean_input)
        next_clean = tokenizer.decode(clean_logits[0, -1].argmax())
        clean_input += next_clean
        clean_tokens.append(next_clean)

        ablated_logits = model.run_with_hooks(
            ablated_input,
            fwd_hooks=[("blocks.3.hook_resid_post", partial(ablation_hook, feature_idx=feature_idx))]
        )
        next_ablated = tokenizer.decode(ablated_logits[0, -1].argmax())
        ablated_input += next_ablated
        ablated_tokens.append(next_ablated)

    print(f"Clean:   {clean_input}")
    print(f"Ablated: {ablated_input}")

generate_with_ablation(text, IDX)

Clean:   LONDON (Reuters) - The British government announced a new deal to buy the company from the British government in the UK, which would allow the company
Ablated: LONDON (Reuters) - The British government announced a new deal to buy the company from the British government in the UK, which would allow the company
